In [ ]:
import time
import random
import math
from   typing import Iterable, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from   torch.utils.data import Dataset, DataLoader, Sampler, random_split

import numpy as np
import h5py
import matplotlib.pyplot as plt

print(f"Cuda Check: {torch.cuda.is_available()}")

# barebone dataset class -------------------------------------------------------
class MyDataset(Dataset):
    """ a minimal dataset class needs at least the three functions __init__, __len__ and __getitem__. This is, in it's simplest form, basically just a lean wrapper for your data, that ensures the one-by-one retrieval of samples. """
    
    def __init__(self, data):
        """ just the constructor for the dataset. Initialize the directories to data / data itself, transforms, etc """
        self.data = data

    def __len__(self):
        """ very standard, just needs to return the total length of the dataset """
        return len(self.data)

    def __getitem__(self, idx):
        """ core method: loads and returns one sample given and index, can grab it directly from data if it's already loaded in memory, or load it from the disk if path is specified. Sample can be manipulated and transformed """
        
        return self.data[idx] 
# TODO: Explore Transforms

# custom sampler ---------------------------------------------------------------
class MySampler(Sampler):
    """ responsible for drawing samples (just indices) from the dataset, also shuffling, repetition, etc. Minimal implementation needs to contain a __iter__ function that yields dataset indices and ideally a __len__ method (number of batches yielded!). This is a simple batched sampler"""
    
    def __init__(self, data_source, batch_size, shuffle=True, drop_last=False):
        self.data_source = data_source
        self.batch_size  = batch_size
        self.shuffle     = shuffle
        self.drop_last   = drop_last
        self.num_samples = len(self.data_source)
        self.indices     = list(range(self.num_samples)) # create a base list of dataset sample indices. TODO: maybe bad to keep this persistent?
    
    def __iter__(self):
        # this is called for every epoch, every time an iterator is "returned"
        if self.shuffle:
            random.shuffle(self.indices) # reshuffles indices at the beginning of each epoch
            
        for start_idx in range(0, self.num_samples, self.batch_size):
            batch = self.indices[start_idx:start_idx+self.batch_size]
            if len(batch) < self.batch_size and self.drop_last:
                continue # skip the last partial batch
            yield batch
                
                
    def __len__(self):
        if self.drop_last:
            return math.floor(self.num_samples / self.batch_size)
        else:
            return math.ceil(self.num_samples / self.batch_size)




class ScheduledBatchSampler(Sampler):
    """
    batch_sampler with batch size scheduling based on a custom dict. Use as custom batch_sampler fn in Dataloader().
    batch schedule = {how many epochs : batch size, ... : ...}
    """
    
    def __init__(self, data_source, schedule, shuffle=False, drop_last=False):
        """
        Args:
            data_source (Dataset): Dataset to sample from.
            schedule (dict)      : mapping batch size to nr of epochs e.g. {128: 10, 256: 20, 512: 50}. no duplicates!
            shuffle (bool)       : whether to shuffle the dataset indices.
            drop_last (bool)     : whether to drop the last incomplete batch.
        """
        
        # set up instance variables
        self.data_source   = data_source                     # this is essentially the whole dataset
        self.num_samples   = len(data_source)                # length of the whole dataset
        self.indices       = list(range(self.num_samples))   # base indices. (shuffled) batches of these are returned
        self.schedule      = self._expand_schedule(schedule) # schedule dict but now with each batch size repeated * n
        self.current_epoch = 0                               # to keep track of the current epoch in training loop
        self.shuffle       = shuffle
        self.drop_last     = drop_last

    def _expand_schedule(self, schedule):
        """
        Expand the schedule to have a list of repeated batch size values. Returns a dict.
        e.g. {128 : 3, 256 : 2} -> {0 : 128, 1 : 128, 2 : 128, 3 : 256, 4 : 256}
        """
        
        expanded_schedule = {}
        cumulative_epochs = 0
        # dict.items() yields a tuple of (key, value)
        for bs, nr_eps in schedule.items():
            for i in range(cumulative_epochs, cumulative_epochs + nr_eps):
                # rebuilds the schedule dict. key i is the epoch nr i and value is the batch size for that epoch
                expanded_schedule[i] = bs
            cumulative_epochs += nr_eps
            
        return expanded_schedule

    def set_epoch(self, epoch):
        """
        update the current epoch when called in the training loop
        note: could also be replaced with something like step() to just increment the epoch by one ...
        """
        
        self.current_epoch = epoch

    def _get_batch_size(self):
        """
        just convenience for getting the currend batch size from the expanded schedule depending on current epoch. If the epoch is larger than what is specified in the schedule, it just returns the last value.
        """
        
        # use dict.get(key) instead of dict[key] in order to be able to specify a default for when key is missing!
        return self.schedule.get(self.current_epoch, list(self.schedule.values())[-1])

    def __iter__(self):
        """
        this function is essentially responsible for returning a generator object. The yield statement is kind of like a return, but instead it just advances the loop further every time the next() method is called, or in a for loop. it kind of remebers the last time it was called.
        """
        
        # determine batch size
        batch_size = self._get_batch_size()
        
        # Shuffle before returning the generator object for the new epoch
        if self.shuffle:
            random.shuffle(self.indices) 

        # determines the start indices for each batch. range(start, stop, step). 
        # Note this is essentially only the index to find the indices in self.indices for one batch xD
        for start_idx in range(0, len(self.indices), batch_size): #TODO: replace len() with just self.num_samples??
            # grab all the indices (now those indices corresponding to samples) belonging to a back
            batch = self.indices[start_idx:start_idx + batch_size]
            # skip yielding the last semi-full batch if self.drop_last is true
            if len(batch) < batch_size and self.drop_last:
                continue
            # basically returns one element when it's called and then pauses the function execution until next()
            yield batch

    def __len__(self):
        """
        just returns the number of batches in the current epoch. changes by 1 depending on whether the last semi-full one is dropped or not.
        """
        
        batch_size = self._get_batch_size()
        if self.drop_last:
            return math.floor(self.num_samples / batch_size)
        else:
            return math.ceil(self.num_samples / batch_size)

# collate function -------------------------------------------------------------

# dataloader -------------------------------------------------------------------
""" combines dataset and sampler and actually returns an iterable dataset with shuffling, batching and memory handling. Lot of low-level memory handling, so I guess it's usually not necessary to touch this. """



with h5py.File("dataset.h5", "r") as file:
    data = file["data"][...]
    
dataset = MyDataset(data)


Cuda Check: True
